In [1]:
import math
import time
import numpy as np
import pandas as pd
import yfinance as yf

def gerar_carteiras(acoes, passo, investimento=None, start="2023-01-01", end="2026-06-30", limite=100_000):
    """
    Baixa dados do yfinance, calcula médias e covariâncias e gera todas as
    carteiras possíveis com pesos que somam 1. Retorna a carteira ótima (maior FO).

    Parâmetros:
        acoes        : lista de tickers (ex: ["PETR3.SA", "VALE3.SA"])
        passo        : tamanho do passo em % inteira (ex: 1, 5, 10)
        investimento : valor total a investir em reais (opcional)
        start        : data de início (padrão: "2023-01-01")
        end          : data de fim   (padrão: "2026-06-30")
        limite       : número máximo de carteiras permitido (padrão: 100_000)

    Retorna:
        df_carteiras   : DataFrame com pesos, retorno, risco e fo de todas as carteiras
        carteira_otima : Series com a carteira de maior fo
        alocacao       : DataFrame com valor (R$) a investir em cada ação (None se investimento não informado)
    """
    n_acoes = len(acoes)
    passos  = 100 // passo
    n_carteiras = math.comb(passos + n_acoes - 1, n_acoes - 1)

    if n_carteiras > limite:
        raise ValueError(
            f"Configuração inválida: {n_acoes} ações com passo={passo}% geraria "
            f"{n_carteiras:,} carteiras, acima do limite de {limite:,}. "
            f"Aumente o passo ou reduza o número de ações."
        )

    print("Baixando dados do yfinance...")
    df = yf.download(tickers=acoes, start=start, end=end, auto_adjust=True)["Close"]
    df_var = df.pct_change(axis=0)
    medias = df_var.mean(axis=0)
    covariancias = df_var.cov()

    colunas = medias.index.tolist()

    def _combinacoes(n, total, passo):
        if n == 1:
            yield (total,)
            return
        for v in range(0, total + 1, passo):
            for resto in _combinacoes(n - 1, total - v, passo):
                yield (v,) + resto

    colecao_carteiras = []
    colecao_retornos  = []
    colecao_riscos    = []
    colecao_fos       = []

    print(f"\n[Timer] Iniciando avaliação ingênua de {n_carteiras:,} carteiras no loop...")
    inicio_calculo = time.time()

    for combo in _combinacoes(n_acoes, 100, passo):
        carteira = pd.Series(list(combo), index=colunas) / 100
        retorno  = carteira @ medias
        risco    = carteira @ covariancias @ carteira
        fo       = retorno / risco

        colecao_carteiras.append(carteira)
        colecao_retornos.append(retorno)
        colecao_riscos.append(risco)
        colecao_fos.append(fo)

    tempo_calculo = time.time() - inicio_calculo
    print(f"[Timer] Avaliação matemática concluída em {tempo_calculo:.6f} segundos.")

    df_carteiras = pd.DataFrame(colecao_carteiras)
    df_carteiras["retorno"] = colecao_retornos
    df_carteiras["risco"]   = colecao_riscos
    df_carteiras["fo"]      = colecao_fos

    carteira_otima = df_carteiras.loc[df_carteiras["fo"].idxmax()]

    alocacao = None
    if investimento is not None:
        pesos = carteira_otima[colunas]
        preco_atual = df.iloc[-1]  # último preço disponível
        valor_por_acao = pesos * investimento
        qtd_por_acao = (valor_por_acao / preco_atual).apply(np.floor)  # arredonda para baixo

        alocacao = pd.DataFrame({
            "peso (%)":        (pesos * 100).round(2),
            "preço atual (R$)": preco_atual.round(2),
            "valor alocado (R$)": valor_por_acao.round(2),
            "quantidade":      qtd_por_acao.astype(int),
            "valor real (R$)": (qtd_por_acao * preco_atual).round(2),
        })
        alocacao.loc["TOTAL"] = [
            100.0,
            None,
            alocacao["valor alocado (R$)"].sum().round(2),
            None,
            alocacao["valor real (R$)"].sum().round(2),
        ]

    return df_carteiras, carteira_otima, alocacao


# Exemplo de uso
if __name__ == "__main__":
    t0 = time.time()
    df_carteiras, carteira_otima, alocacao = gerar_carteiras(
        acoes=["PETR3.SA", "VALE3.SA", "EMBJ3.SA"],
        passo=5,
        investimento=10_000,
    )
    t1 = time.time()
    
    print(f"\nTempo total da execução inteira: {t1 - t0:.2f} segundos.")
    
    try:
        display(df_carteiras)
        print("\nCarteira ótima (maior FO):")
        display(carteira_otima)
        print("\nAlocação do investimento:")
        display(alocacao)
    except NameError:
        print("\nFunção 'display' não disponível fora de um notebook Jupyter. Usando 'print':")
        print("\nCarteira ótima (maior FO):")
        print(carteira_otima)

Baixando dados do yfinance...


[*********************100%***********************]  3 of 3 completed


[Timer] Iniciando avaliação ingênua de 231 carteiras no loop...
[Timer] Avaliação matemática concluída em 0.050988 segundos.

Tempo total da execução inteira: 0.69 segundos.


,EMBJ3.SA,PETR3.SA,VALE3.SA,retorno,risco,fo
0,0.00,0.00,1.00,0.000334,0.000250,1.336509
1,0.00,0.05,0.95,0.000387,0.000231,1.674577
2,0.00,0.10,0.90,0.000440,0.000215,2.048382
3,0.00,0.15,0.85,0.000493,0.000201,2.452755
4,0.00,0.20,0.80,0.000546,0.000190,2.878492
...,...,...,...,...,...,...
226,0.90,0.05,0.05,0.002150,0.000491,4.381611
227,0.90,0.10,0.00,0.002203,0.000485,4.538203
228,0.95,0.00,0.05,0.002195,0.000548,4.001962
229,0.95,0.05,0.00,0.002248,0.000541,4.152531



Carteira ótima (maior FO):


EMBJ3.SA    0.350000
PETR3.SA    0.450000
VALE3.SA    0.200000
retorno     0.001497
risco       0.000159
fo          9.422408
Name: 135, dtype: float64


Alocação do investimento:


,peso (%),preço atual (R$),valor alocado (R$),quantidade,valor real (R$)
EMBJ3.SA,35.0,78.74,3500.0,44.0,3464.56
PETR3.SA,45.0,43.07,4500.0,104.0,4479.28
VALE3.SA,20.0,79.78,2000.0,25.0,1994.50
TOTAL,100.0,NaN,10000.0,NaN,9938.34
